In [12]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
import os

In [13]:
load_dotenv(find_dotenv())

True

In [14]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [15]:
# solving a quadratic equation (conditional because D decides the nature and value of the roots)

class QuadState(TypedDict):
    a: int
    b: int
    c: int

    equation: str
    discriminant: str
    result: str

In [23]:
def show_equation(state: QuadState):
    equation = f'{state["a"]}x2{state["b"]}x{state["c"]}'
    return {'equation': equation}

def calculate_discriminant(state: QuadState):
    return {"discriminant": state["b"]**2 - (4 * state["a"] * state["c"])}

def real_roots(state: QuadState):
    root1 = (-state['b'] + state["discriminant"]**0.5)/(2*state["a"])
    root2 = (-state['b'] - state["discriminant"]**0.5)/(2*state["a"])
    result = f'The roots are {root1} and {root2}'
    return {'result': result}

def repeated_roots(state: QuadState):
    root = (-state['b']/(2*state["a"]*state["c"]))
    result = f'The root is {root}'
    return {'result': result}

def imaginary_roots(state: QuadState):
    return {'result': f'No real roots'}

def check_condition(state: QuadState) -> Literal["real_roots", "repeated_roots", "imaginary_roots"]:
    if state['discriminant'] > 0:
        return "real_roots"
    elif state["discriminant"] == 0:
        return "repeated_roots"
    else:
        return "imaginary_roots"

In [24]:
graph = StateGraph(QuadState)

graph.add_node("show_equation", show_equation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("real_roots", real_roots)
graph.add_node("repeated_roots", repeated_roots)
graph.add_node("imaginary_roots", imaginary_roots)

graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")

# add conditional edge
graph.add_conditional_edges("calculate_discriminant", check_condition)
graph.add_edge("real_roots", END)
graph.add_edge("repeated_roots", END)
graph.add_edge("imaginary_roots", END)

workflow = graph.compile()

In [ ]:
initial_state = {
    'a': 4,
    'b': -5,
    'c': -4
}

workflow.invoke(initial_state)

{'a': 4,
 'b': -5,
 'c': -4,
 'equation': '4x2-5x-4',
 'discriminant': 89,
 'result': 'The roots are 1.8042476415070754 and -0.5542476415070754'}